# Chapter 7.6: Explanation & Reasoning

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement chain-of-thought reasoning for recommendation decisions
2. Generate feature-based, review-based, and knowledge-based explanations
3. Build natural language explanation pipelines using templates and reasoning
4. Measure user trust and explanation effectiveness
5. Create counterfactual explanations for recommendations
6. Compare different explanation styles and their impact on user behavior
7. Understand the trade-offs between explanation quality and computational cost

## Prerequisites

- Basic recommendation system concepts
- Understanding of feature importance and model interpretability
- Familiarity with natural language generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part7/chapter_7.6_explanation.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part7/chapter_7.6_explanation.ipynb)

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict

np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. Why Explanations Matter in Recommendation

Explainable recommendation (Zhang & Chen, 2020) addresses three key questions:

1. **Why this item?** — Justify the recommendation
2. **How was it chosen?** — Transparency about the process
3. **What if?** — Counterfactual reasoning

Benefits of explanations:
- **Trust**: Users trust recommendations they understand
- **Satisfaction**: Explained recs lead to higher satisfaction
- **Effectiveness**: Users make better decisions with explanations
- **Persuasion**: Well-explained recs increase click-through rates

### Explanation Taxonomy

| Type | Example | Method |
|------|---------|--------|
| Feature-based | "Because you liked action movies" | Feature attribution |
| Review-based | "Users said: 'Great soundtrack!'" | Review extraction |
| Social | "Users like you also watched..." | Collaborative filtering |
| Knowledge-based | "Directed by your favorite director" | Knowledge graph |
| Counterfactual | "If you liked drama, we'd recommend X instead" | Counterfactual reasoning |

> **💡 Concept:** The best explanation style depends on the domain. E-commerce benefits from feature-based explanations, while movie recommendations benefit from social proof and review-based explanations.

In [ ]:
# Synthetic data for explanation experiments
@dataclass
class Movie:
    movie_id: int
    title: str
    genres: List[str]
    director: str
    year: int
    avg_rating: float
    n_ratings: int
    features: Dict[str, float]  # Feature scores (0-1)
    keywords: List[str]

@dataclass
class Review:
    user_id: int
    movie_id: int
    rating: float
    text: str
    helpful_votes: int

@dataclass
class UserProfile:
    user_id: int
    preferred_genres: List[str]
    preferred_directors: List[str]
    watch_history: List[int]
    ratings: Dict[int, float]
    feature_preferences: Dict[str, float]  # Feature name -> preference weight


def create_movie_database():
    rng = np.random.RandomState(42)
    genres_pool = ['Action', 'Comedy', 'Drama', 'Sci-Fi', 'Thriller', 'Romance', 'Horror', 'Animation']
    directors = ['Director_A', 'Director_B', 'Director_C', 'Director_D', 'Director_E']
    feature_names = ['acting', 'visuals', 'story', 'soundtrack', 'pacing', 'originality']
    keyword_pool = ['epic', 'emotional', 'thrilling', 'funny', 'dark', 'inspiring', 'slow-burn', 
                    'fast-paced', 'thought-provoking', 'heartwarming', 'intense', 'quirky']
    
    review_templates = {
        5: ["Amazing! {feature} was outstanding.", "A masterpiece. Loved the {feature}.",
            "Best {genre} film I've seen. Incredible {feature}."],
        4: ["Really good. The {feature} was impressive.", "Solid film. {feature} stood out.",
            "Enjoyed it. Great {feature} and {feature2}."],
        3: ["Decent. {feature} was okay.", "Average {genre} film. {feature} could be better."],
        2: ["Disappointing. {feature} was weak.", "Expected more from the {feature}."],
        1: ["Terrible. {feature} was awful.", "Waste of time. Bad {feature}."]
    }
    
    movies = []
    all_reviews = []
    
    for i in range(60):
        n_genres = rng.randint(1, 4)
        genres = list(rng.choice(genres_pool, n_genres, replace=False))
        features = {f: round(rng.beta(2, 2), 2) for f in feature_names}
        n_keywords = rng.randint(2, 5)
        keywords = list(rng.choice(keyword_pool, n_keywords, replace=False))
        avg_rating = round(rng.uniform(2.0, 5.0), 1)
        
        movie = Movie(
            movie_id=i,
            title=f"Movie_{i}_{genres[0]}",
            genres=genres,
            director=rng.choice(directors),
            year=int(rng.uniform(2015, 2025)),
            avg_rating=avg_rating,
            n_ratings=int(rng.exponential(100)),
            features=features,
            keywords=keywords
        )
        movies.append(movie)
        
        # Generate reviews
        for u in range(min(movie.n_ratings, 10)):
            r = max(1, min(5, int(avg_rating + rng.normal(0, 0.8))))
            templates = review_templates[r]
            template = rng.choice(templates)
            text = template.format(
                feature=rng.choice(feature_names),
                feature2=rng.choice(feature_names),
                genre=genres[0]
            )
            all_reviews.append(Review(
                user_id=u, movie_id=i, rating=float(r),
                text=text, helpful_votes=int(rng.exponential(5))
            ))
    
    # Create users
    users = []
    for u in range(20):
        prefs = list(rng.choice(genres_pool, rng.randint(2, 4), replace=False))
        dirs = list(rng.choice(directors, rng.randint(1, 3), replace=False))
        feat_prefs = {f: round(rng.uniform(-0.5, 1.0), 2) for f in feature_names}
        history = list(rng.choice(60, rng.randint(5, 15), replace=False))
        ratings = {int(mid): round(rng.uniform(2, 5), 1) for mid in history}
        
        users.append(UserProfile(
            user_id=u,
            preferred_genres=prefs,
            preferred_directors=dirs,
            watch_history=history,
            ratings=ratings,
            feature_preferences=feat_prefs
        ))
    
    return movies, all_reviews, users, feature_names


movies, reviews, users, feature_names = create_movie_database()
print(f"Created {len(movies)} movies, {len(reviews)} reviews, {len(users)} users")
print(f"\nSample movie: {movies[0].title}, genres={movies[0].genres}, rating={movies[0].avg_rating}")

## 2. Feature-Based Explanations

Feature-based explanations identify which item features contributed most to the recommendation score.

Given a scoring function $f(u, i) = \sum_k w_k^u \cdot v_k^i$, the contribution of feature $k$ is:

$$\text{contribution}_k = w_k^u \cdot v_k^i$$

The explanation highlights the top contributing features.

In [ ]:
class FeatureBasedExplainer:
    """Generate feature-based explanations for recommendations."""
    
    def __init__(self, feature_names: List[str]):
        self.feature_names = feature_names
    
    def score_item(self, user: UserProfile, movie: Movie) -> Tuple[float, Dict[str, float]]:
        """Score item with per-feature contributions."""
        contributions = {}
        total = 0.0
        
        # Feature-based score
        for feat in self.feature_names:
            user_weight = user.feature_preferences.get(feat, 0.0)
            item_score = movie.features.get(feat, 0.0)
            contrib = user_weight * item_score
            contributions[feat] = contrib
            total += contrib
        
        # Genre match bonus
        genre_overlap = len(set(user.preferred_genres) & set(movie.genres))
        genre_contrib = genre_overlap * 0.3
        contributions['genre_match'] = genre_contrib
        total += genre_contrib
        
        # Director match bonus
        if movie.director in user.preferred_directors:
            contributions['director_match'] = 0.5
            total += 0.5
        else:
            contributions['director_match'] = 0.0
        
        return total, contributions
    
    def explain(self, user: UserProfile, movie: Movie, top_k: int = 3) -> Dict:
        """Generate feature-based explanation."""
        score, contributions = self.score_item(user, movie)
        
        # Sort by contribution
        sorted_contribs = sorted(contributions.items(), key=lambda x: -abs(x[1]))
        top_positive = [(f, c) for f, c in sorted_contribs if c > 0][:top_k]
        top_negative = [(f, c) for f, c in sorted_contribs if c < 0][:2]
        
        # Generate text
        reasons = []
        for feat, contrib in top_positive:
            if feat == 'genre_match':
                matching = set(user.preferred_genres) & set(movie.genres)
                reasons.append(f"Matches your favorite genres: {', '.join(matching)}")
            elif feat == 'director_match':
                reasons.append(f"Directed by {movie.director}, one of your preferred directors")
            else:
                reasons.append(f"Strong {feat} (score: {movie.features[feat]:.1f}) which you value highly")
        
        warnings = []
        for feat, contrib in top_negative:
            if feat not in ('genre_match', 'director_match'):
                warnings.append(f"{feat} may not meet your standards")
        
        return {
            'score': score,
            'contributions': dict(sorted_contribs),
            'reasons': reasons,
            'warnings': warnings,
            'text': self._format_explanation(movie, reasons, warnings)
        }
    
    def _format_explanation(self, movie: Movie, reasons: List[str], 
                           warnings: List[str]) -> str:
        text = f"We recommend '{movie.title}' because:\n"
        for i, reason in enumerate(reasons, 1):
            text += f"  {i}. {reason}\n"
        if warnings:
            text += "Note: " + "; ".join(warnings) + "\n"
        return text


# Demo
explainer = FeatureBasedExplainer(feature_names)
user = users[0]

# Get recommendations
movie_scores = []
for movie in movies:
    if movie.movie_id not in user.watch_history:
        score, _ = explainer.score_item(user, movie)
        movie_scores.append((movie, score))

movie_scores.sort(key=lambda x: -x[1])

print(f"User preferences: genres={user.preferred_genres}, directors={user.preferred_directors}")
print(f"Feature preferences: {dict(sorted(user.feature_preferences.items(), key=lambda x: -x[1])[:3])}")
print()

for movie, score in movie_scores[:3]:
    explanation = explainer.explain(user, movie)
    print(explanation['text'])
    print(f"  Contribution breakdown: {dict(list(explanation['contributions'].items())[:4])}")
    print()

## 3. Chain-of-Thought Reasoning for Recommendations

Chain-of-Thought (CoT) prompting (Wei et al., 2022 - Google) makes the reasoning process explicit:

```
Step 1: Identify user preferences -> likes Sci-Fi, values story
Step 2: Find matching items -> Movie_5 is Sci-Fi with strong story
Step 3: Check quality -> 4.5/5.0 rating, 200+ reviews
Step 4: Compare alternatives -> Movie_5 beats Movie_12 on story
Step 5: Final recommendation with justification
```

> **💡 Concept:** CoT reasoning doesn't just explain the final answer; it shows the **reasoning process**. This is more transparent and helps users understand and trust the recommendation logic.

In [ ]:
class ChainOfThoughtExplainer:
    """Generate chain-of-thought reasoning for recommendations."""
    
    def __init__(self, movies: List[Movie], reviews: List[Review]):
        self.movies = movies
        self.movie_index = {m.movie_id: m for m in movies}
        self.reviews_by_movie = defaultdict(list)
        for r in reviews:
            self.reviews_by_movie[r.movie_id].append(r)
    
    def reason(self, user: UserProfile, candidates: List[Movie], 
               top_k: int = 3) -> Dict:
        """Generate step-by-step reasoning."""
        steps = []
        
        # Step 1: Analyze user preferences
        step1 = self._analyze_preferences(user)
        steps.append(step1)
        
        # Step 2: Filter by genre and director preferences
        filtered, step2 = self._filter_candidates(user, candidates)
        steps.append(step2)
        
        # Step 3: Score by feature alignment
        scored, step3 = self._score_by_features(user, filtered)
        steps.append(step3)
        
        # Step 4: Verify with reviews
        verified, step4 = self._verify_with_reviews(scored[:top_k * 2])
        steps.append(step4)
        
        # Step 5: Final selection
        final, step5 = self._final_selection(verified, top_k)
        steps.append(step5)
        
        # Compile full reasoning chain
        full_reasoning = "\n".join([
            f"Step {i+1}: {step['title']}\n  {step['reasoning']}"
            for i, step in enumerate(steps)
        ])
        
        return {
            'steps': steps,
            'recommendations': final,
            'full_reasoning': full_reasoning
        }
    
    def _analyze_preferences(self, user: UserProfile) -> Dict:
        top_features = sorted(user.feature_preferences.items(), key=lambda x: -x[1])[:3]
        return {
            'title': 'Analyze User Preferences',
            'reasoning': (
                f"User prefers genres: {user.preferred_genres}. "
                f"Favorite directors: {user.preferred_directors}. "
                f"Values: {', '.join(f'{f} ({s:.2f})' for f, s in top_features)}. "
                f"Has watched {len(user.watch_history)} movies."
            )
        }
    
    def _filter_candidates(self, user: UserProfile, 
                           candidates: List[Movie]) -> Tuple[List[Movie], Dict]:
        filtered = []
        for m in candidates:
            if m.movie_id in user.watch_history:
                continue
            genre_match = len(set(user.preferred_genres) & set(m.genres)) > 0
            if genre_match or m.avg_rating >= 4.0:
                filtered.append(m)
        
        return filtered, {
            'title': 'Filter Candidates',
            'reasoning': (
                f"From {len(candidates)} candidates, filtered to {len(filtered)} "
                f"based on genre preference match or high rating (>=4.0). "
                f"Excluded {len(user.watch_history)} already watched movies."
            )
        }
    
    def _score_by_features(self, user: UserProfile, 
                           candidates: List[Movie]) -> Tuple[List[Tuple], Dict]:
        scored = []
        for m in candidates:
            score = 0.0
            for feat, pref in user.feature_preferences.items():
                score += pref * m.features.get(feat, 0)
            # Genre bonus
            score += len(set(user.preferred_genres) & set(m.genres)) * 0.3
            # Director bonus
            if m.director in user.preferred_directors:
                score += 0.5
            scored.append((m, score))
        
        scored.sort(key=lambda x: -x[1])
        
        top3 = scored[:3]
        return scored, {
            'title': 'Score by Feature Alignment',
            'reasoning': (
                f"Scored {len(candidates)} candidates by feature alignment. "
                f"Top 3: {', '.join(f'{m.title} ({s:.2f})' for m, s in top3)}."
            )
        }
    
    def _verify_with_reviews(self, scored: List[Tuple]) -> Tuple[List[Tuple], Dict]:
        verified = []
        review_insights = []
        
        for m, score in scored:
            movie_reviews = self.reviews_by_movie.get(m.movie_id, [])
            if movie_reviews:
                avg_review_rating = np.mean([r.rating for r in movie_reviews])
                top_review = max(movie_reviews, key=lambda r: r.helpful_votes)
                review_boost = (avg_review_rating - 3.0) / 2.0 * 0.2
                verified.append((m, score + review_boost, top_review))
                review_insights.append(f"{m.title}: avg review {avg_review_rating:.1f}, top: '{top_review.text}'")
            else:
                verified.append((m, score, None))
        
        verified.sort(key=lambda x: -x[1])
        
        return verified, {
            'title': 'Verify with Reviews',
            'reasoning': (
                f"Cross-checked with user reviews. "
                + (" ".join(review_insights[:3]) if review_insights else "No reviews available.")
            )
        }
    
    def _final_selection(self, verified: List[Tuple], top_k: int) -> Tuple[List, Dict]:
        final = []
        selected_genres = set()
        
        for m, score, review in verified:
            # Add diversity: don't select too many from same genre
            m_genres = set(m.genres)
            if len(selected_genres & m_genres) < 2:
                final.append({'movie': m, 'score': score, 'top_review': review})
                selected_genres |= m_genres
            if len(final) >= top_k:
                break
        
        return final, {
            'title': 'Final Selection with Diversity',
            'reasoning': (
                f"Selected top {len(final)} movies with genre diversity. "
                f"Genres covered: {', '.join(selected_genres)}. "
                f"Final picks: {', '.join(r['movie'].title for r in final)}."
            )
        }


# Demo chain-of-thought
cot_explainer = ChainOfThoughtExplainer(movies, reviews)
user = users[0]
result = cot_explainer.reason(user, movies, top_k=3)

print("=== Chain-of-Thought Reasoning ===")
print(result['full_reasoning'])
print("\n=== Final Recommendations ===")
for rec in result['recommendations']:
    m = rec['movie']
    print(f"  {m.title} ({m.year}) | {', '.join(m.genres)} | Score: {rec['score']:.3f}")
    if rec['top_review']:
        print(f"    Top review: '{rec['top_review'].text}'")

## 4. Counterfactual Explanations

Counterfactual explanations answer: "What would have to change for a different recommendation?"

$$\text{Counterfactual}: \text{If } x_k \text{ were } x_k', \text{ then } f(x) \text{ would be } y'$$

Examples:
- "If you had rated Drama higher, we would recommend Movie X"
- "If this movie had better pacing, it would rank higher"
- "You'd like this movie if you enjoy slow-burn thrillers"

> **🔑 Pro Tip:** Counterfactual explanations are particularly useful for borderline recommendations (items that almost made the list) and for helping users understand how to get better recommendations.

In [ ]:
class CounterfactualExplainer:
    """Generate counterfactual explanations for recommendations."""
    
    def __init__(self, feature_names: List[str]):
        self.feature_names = feature_names
    
    def explain_why_not(self, user: UserProfile, movie: Movie,
                        recommended_movies: List[Movie]) -> Dict:
        """Explain why a movie was NOT recommended."""
        # Score the movie
        score = self._score(user, movie)
        rec_scores = [(m, self._score(user, m)) for m in recommended_movies]
        threshold = min(s for m, s in rec_scores)
        gap = threshold - score
        
        counterfactuals = []
        
        # Feature-based counterfactuals
        for feat in self.feature_names:
            user_pref = user.feature_preferences.get(feat, 0)
            item_val = movie.features.get(feat, 0)
            
            if user_pref > 0 and item_val < 0.5:
                needed = min(1.0, item_val + gap / max(user_pref, 0.01))
                if needed <= 1.0:
                    counterfactuals.append({
                        'type': 'item_feature',
                        'feature': feat,
                        'current': item_val,
                        'needed': round(needed, 2),
                        'text': f"If '{movie.title}' had better {feat} (from {item_val:.1f} to {needed:.1f}), it would be recommended."
                    })
        
        # Genre-based counterfactual
        if not set(user.preferred_genres) & set(movie.genres):
            counterfactuals.append({
                'type': 'user_preference',
                'text': f"If you liked {movie.genres[0]}, '{movie.title}' would be recommended."
            })
        
        # Director-based counterfactual
        if movie.director not in user.preferred_directors:
            counterfactuals.append({
                'type': 'user_preference',
                'text': f"If {movie.director} were among your preferred directors, this would rank higher."
            })
        
        return {
            'movie': movie,
            'score': score,
            'threshold': threshold,
            'gap': gap,
            'counterfactuals': counterfactuals
        }
    
    def explain_what_if(self, user: UserProfile, movie: Movie) -> List[str]:
        """Generate 'what if' scenarios for a recommended movie."""
        what_ifs = []
        
        # What if user preferences change?
        for feat in self.feature_names:
            pref = user.feature_preferences.get(feat, 0)
            val = movie.features.get(feat, 0)
            if pref > 0.3 and val > 0.5:
                what_ifs.append(
                    f"If your preference for {feat} decreases, this recommendation would drop in ranking."
                )
            elif pref < 0 and val > 0.7:
                what_ifs.append(
                    f"Despite your low preference for {feat}, this movie excels at it ({val:.1f}/1.0). You might discover you enjoy it."
                )
        
        return what_ifs[:3]
    
    def _score(self, user: UserProfile, movie: Movie) -> float:
        score = 0.0
        for feat in self.feature_names:
            score += user.feature_preferences.get(feat, 0) * movie.features.get(feat, 0)
        score += len(set(user.preferred_genres) & set(movie.genres)) * 0.3
        if movie.director in user.preferred_directors:
            score += 0.5
        return score


# Demo counterfactual explanations
cf_explainer = CounterfactualExplainer(feature_names)
user = users[0]

# Get recommended movies
scored = [(m, cf_explainer._score(user, m)) for m in movies if m.movie_id not in user.watch_history]
scored.sort(key=lambda x: -x[1])
recommended = [m for m, s in scored[:3]]
not_recommended = [m for m, s in scored[10:13]]  # Some non-recommended movies

print("=== Why Not Recommended? ===")
for movie in not_recommended[:2]:
    result = cf_explainer.explain_why_not(user, movie, recommended)
    print(f"\n'{movie.title}' (score: {result['score']:.3f}, threshold: {result['threshold']:.3f})")
    for cf in result['counterfactuals'][:2]:
        print(f"  -> {cf['text']}")

print("\n=== What If Scenarios ===")
for movie in recommended[:2]:
    what_ifs = cf_explainer.explain_what_if(user, movie)
    print(f"\n'{movie.title}':")
    for wi in what_ifs:
        print(f"  -> {wi}")

## 5. Measuring Explanation Quality

How do we evaluate explanation quality? Key metrics:

1. **Fidelity**: Does the explanation accurately reflect the model's reasoning?
2. **Comprehensibility**: Can users understand the explanation?
3. **Persuasiveness**: Does the explanation increase acceptance?
4. **Diversity**: Are explanations varied enough?
5. **Actionability**: Can users act on the explanation?

In [ ]:
class ExplanationEvaluator:
    """Evaluate quality of recommendation explanations."""
    
    def __init__(self):
        pass
    
    def evaluate_fidelity(self, explanation: Dict, true_contributions: Dict) -> float:
        """Does the explanation match actual feature contributions?"""
        explained_features = set()
        for reason in explanation.get('reasons', []):
            for feat in true_contributions:
                if feat in reason.lower():
                    explained_features.add(feat)
        
        # Top contributing features
        top_true = sorted(true_contributions.items(), key=lambda x: -abs(x[1]))[:3]
        top_true_feats = {f for f, c in top_true if c > 0}
        
        if not top_true_feats:
            return 1.0
        
        overlap = len(explained_features & top_true_feats)
        return overlap / len(top_true_feats)
    
    def evaluate_comprehensibility(self, explanation_text: str) -> float:
        """Proxy for comprehensibility based on text properties."""
        words = explanation_text.split()
        n_words = len(words)
        
        # Readability heuristics
        score = 1.0
        
        # Penalize too long or too short
        if n_words > 100:
            score -= 0.3
        elif n_words < 10:
            score -= 0.2
        
        # Reward structured format (numbered lists, bullets)
        if any(c in explanation_text for c in ['1.', '2.', '-', '*']):
            score += 0.1
        
        # Penalize jargon
        jargon = ['embedding', 'cosine', 'sigmoid', 'latent', 'vector']
        jargon_count = sum(1 for w in words if w.lower() in jargon)
        score -= jargon_count * 0.1
        
        return max(0, min(1, score))
    
    def evaluate_persuasiveness(self, explanation: Dict, 
                                 user: UserProfile) -> float:
        """Simulate how persuasive the explanation is."""
        score = 0.0
        reasons = explanation.get('reasons', [])
        
        for reason in reasons:
            # Social proof
            if 'review' in reason.lower() or 'users' in reason.lower():
                score += 0.3
            # Personal relevance
            if any(g in reason for g in user.preferred_genres):
                score += 0.3
            # Quality signals
            if 'rating' in reason.lower() or 'top' in reason.lower():
                score += 0.2
            # Director/authority
            if 'director' in reason.lower():
                score += 0.2
        
        return min(1, score)
    
    def evaluate_diversity(self, explanations: List[Dict]) -> float:
        """Are explanations diverse across recommendations?"""
        if len(explanations) < 2:
            return 1.0
        
        reason_sets = []
        for expl in explanations:
            reason_words = set()
            for reason in expl.get('reasons', []):
                reason_words.update(reason.lower().split())
            reason_sets.append(reason_words)
        
        # Average pairwise dissimilarity
        dissimilarities = []
        for i in range(len(reason_sets)):
            for j in range(i + 1, len(reason_sets)):
                intersection = len(reason_sets[i] & reason_sets[j])
                union = len(reason_sets[i] | reason_sets[j])
                jaccard = intersection / max(union, 1)
                dissimilarities.append(1 - jaccard)
        
        return np.mean(dissimilarities) if dissimilarities else 1.0


# Evaluate different explanation styles
evaluator = ExplanationEvaluator()
user = users[0]

# Generate explanations for top movies
feature_explainer = FeatureBasedExplainer(feature_names)
explanations = []
fidelities = []
comprehensibilities = []
persuasiveness_scores = []

for movie, _ in scored[:5]:
    expl = feature_explainer.explain(user, movie)
    explanations.append(expl)
    
    fidelity = evaluator.evaluate_fidelity(expl, expl['contributions'])
    comprehensibility = evaluator.evaluate_comprehensibility(expl['text'])
    persuasiveness = evaluator.evaluate_persuasiveness(expl, user)
    
    fidelities.append(fidelity)
    comprehensibilities.append(comprehensibility)
    persuasiveness_scores.append(persuasiveness)

diversity = evaluator.evaluate_diversity(explanations)

print("Explanation Quality Metrics (averaged over 5 recommendations):")
print(f"  Fidelity:          {np.mean(fidelities):.3f}")
print(f"  Comprehensibility: {np.mean(comprehensibilities):.3f}")
print(f"  Persuasiveness:    {np.mean(persuasiveness_scores):.3f}")
print(f"  Diversity:         {diversity:.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Per-recommendation metrics
x = range(1, 6)
axes[0].plot(x, fidelities, 'o-', label='Fidelity', color='steelblue')
axes[0].plot(x, comprehensibilities, 's-', label='Comprehensibility', color='coral')
axes[0].plot(x, persuasiveness_scores, '^-', label='Persuasiveness', color='green')
axes[0].set_xlabel('Recommendation Rank')
axes[0].set_ylabel('Score')
axes[0].set_title('Explanation Quality by Rank')
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# Summary radar
metrics = ['Fidelity', 'Comprehensibility', 'Persuasiveness', 'Diversity']
values = [np.mean(fidelities), np.mean(comprehensibilities), 
          np.mean(persuasiveness_scores), diversity]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
values_plot = values + values[:1]
angles_plot = angles + angles[:1]

ax2 = axes[1]
ax2 = fig.add_subplot(122, polar=True)
ax2.plot(angles_plot, values_plot, 'o-', linewidth=2, color='steelblue')
ax2.fill(angles_plot, values_plot, alpha=0.2, color='steelblue')
ax2.set_xticks(angles)
ax2.set_xticklabels(metrics)
ax2.set_title('Explanation Quality Summary')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 6. Summary

| Explanation Type | Strength | Best For |
|-----------------|----------|----------|
| Feature-based | Precise, actionable | E-commerce, tech products |
| Review-based | Social proof | Entertainment, restaurants |
| Chain-of-thought | Transparent process | Complex decisions |
| Counterfactual | Actionable insights | User preference learning |

**Key references:**
- Zhang & Chen (2020) - Explainable Recommendation: A Survey
- Wei et al. (2022) - Chain-of-Thought Prompting (Google)
- Tintarev & Masthoff (2015) - Explaining Recommendations
- Wachter et al. (2017) - Counterfactual Explanations

---

## Exercises

### 🏋️ Exercise 1: Generate Chain-of-Thought Explanations

In [ ]:
# 🏋️ Exercise 1: Enhanced chain-of-thought with user-specific reasoning

class PersonalizedCoTExplainer:
    """CoT explainer that adapts reasoning style to user."""
    
    def __init__(self, movies: List[Movie], reviews: List[Review]):
        self.movies = movies
        self.reviews = reviews
    
    def explain(self, user: UserProfile, movie: Movie, 
                style: str = 'analytical') -> str:
        # TODO: Implement different explanation styles:
        # 'analytical': Focus on features and data
        # 'casual': Conversational tone
        # 'comparative': Compare with movies user has seen
        # 'emotional': Focus on mood and experience
        pass

# TODO: Generate explanations in all 4 styles for the same recommendation
# and compare user simulated acceptance rates

print("Exercise 1: Implement PersonalizedCoTExplainer")

### 🏋️ Exercise 2: Contrastive Explanations

In [ ]:
# 🏋️ Exercise 2: Contrastive explanations ("X over Y because...")

class ContrastiveExplainer:
    """Explain why one movie was recommended over another."""
    
    def __init__(self, feature_names: List[str]):
        self.feature_names = feature_names
    
    def explain_contrast(self, user: UserProfile, 
                          recommended: Movie, rejected: Movie) -> str:
        # TODO: Generate explanation like:
        # "We recommended 'Movie A' over 'Movie B' because:
        #  - Movie A has better story (0.8 vs 0.4), which you value highly
        #  - Movie A matches your preferred genre (Sci-Fi)
        #  - However, Movie B has better visuals (0.9 vs 0.6)"
        pass

print("Exercise 2: Implement ContrastiveExplainer")

### 🏋️ Exercise 3: Explanation A/B Testing

In [ ]:
# 🏋️ Exercise 3: Simulate A/B test of explanation effectiveness

def simulate_explanation_ab_test(
    movies: List[Movie],
    users: List[UserProfile],
    explanation_types: List[str],
    n_trials: int = 1000
) -> Dict:
    # TODO:
    # For each trial:
    #   1. Pick a random user and a recommended movie
    #   2. Generate explanation using each type
    #   3. Simulate user acceptance (higher for better explanations)
    #   4. Track CTR, satisfaction, and trust per explanation type
    # Return comparison metrics
    pass

# TODO: Run A/B test comparing:
# - No explanation
# - Feature-based
# - Review-based  
# - Chain-of-thought
# Plot results

print("Exercise 3: Implement explanation A/B testing")